# Engenharia de Dados - Ovitrampas Recife 2026


## 0. Configuração do ambiente

Importamos as bibliotecas e localizamos a raiz do projeto (independe de onde o notebook
é executado). O arquivo de origem pode estar em formato CSV ou XLSX, e o código
aceita os dois.


In [1]:
import warnings
from pathlib import Path
import re
import sqlite3
import unicodedata

import pandas as pd
warnings.filterwarnings("ignore")


def _achar_raiz() -> Path:
    """Sobe no sistema de pastas até encontrar o diretório do projeto."""
    atual = Path.cwd()
    while atual != atual.parent:
        candidatos = [
            atual / "dados" / "entrada" / "ovitrampas_2026.csv",
            atual / "dados" / "entrada" / "ovitrampas_2026.xlsx",
        ]
        if any(c.exists() for c in candidatos):
            return atual
        atual = atual.parent
    return Path.cwd()


ROOT = _achar_raiz()
CSV_ENTRADA = ROOT / "dados" / "entrada" / "ovitrampas_2026.csv"
XLSX_ENTRADA = ROOT / "dados" / "entrada" / "ovitrampas_2026.xlsx"
ARQUIVO_ENTRADA = CSV_ENTRADA if CSV_ENTRADA.exists() else XLSX_ENTRADA
CAMINHO_BANCO = ROOT / "banco" / "ovitrampas.db"

print("Raiz do projeto :", ROOT)
print("Arquivo entrada  :", ARQUIVO_ENTRADA)
print("Banco SQLite    :", CAMINHO_BANCO)


Raiz do projeto : C:\Users\pedro\OneDrive\Desktop\Eng_dados_ovitrampas
Arquivo entrada  : C:\Users\pedro\OneDrive\Desktop\Eng_dados_ovitrampas\dados\entrada\ovitrampas_2026.csv
Banco SQLite    : C:\Users\pedro\OneDrive\Desktop\Eng_dados_ovitrampas\banco\ovitrampas.db


## 1. Leitura do arquivo CSV

O notebook aceita CSV ou XLSX; a leitura usa o arquivo encontrado na pasta de entrada.


In [2]:
if ARQUIVO_ENTRADA.suffix.lower() == ".csv":
    df = pd.read_csv(ARQUIVO_ENTRADA, sep=";")
else:
    df = pd.read_excel(ARQUIVO_ENTRADA)

print("Dimensão:", df.shape)
print("Colunas :", df.columns.tolist())
df.head()


Dimensão: (31455, 17)
Colunas : ['Ano', 'CICLOS', 'DS', 'BAIRRO', 'BAIRRO_AREA', 'AGENTE / MATRICULA', 'QT', 'ID_OVT', 'DT_COLETA', 'DT_ENTREGA_APOIO', 'DT_ENTREGA_LAB', 'DT_LEITURA', 'NM_TEC_LAB', 'DT_DIG', 'DT_ENVIO_DS', 'N_OVOS', 'OBS']


,Ano,CICLOS,DS,BAIRRO,BAIRRO_AREA,AGENTE / MATRICULA,QT,ID_OVT,DT_COLETA,DT_ENTREGA_APOIO,DT_ENTREGA_LAB,DT_LEITURA,NM_TEC_LAB,DT_DIG,DT_ENVIO_DS,N_OVOS,OBS
0,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1001,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,16,NaN
1,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1002,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,14,NaN
2,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1003,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,83,NaN
3,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1004,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,77,NaN
4,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1005,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,88,NaN


In [3]:
df.tail()

,Ano,CICLOS,DS,BAIRRO,BAIRRO_AREA,AGENTE / MATRICULA,QT,ID_OVT,DT_COLETA,DT_ENTREGA_APOIO,DT_ENTREGA_LAB,DT_LEITURA,NM_TEC_LAB,DT_DIG,DT_ENVIO_DS,N_OVOS,OBS
31450,2026,12,VIII,COHAB,COHAB 4,FLÁVIO JORGE/78.367-0,719,COH8122,2026-07-13,2026-07-15,2026-07-15,2026-07-18,FABIANE,2026-07-20 00:00:00,2026-07-31 00:00:00,0,NaN
31451,2026,12,VIII,COHAB,COHAB 4,DARCIELE CABRAL/130.177-2,713,COH8111,2026-07-13,2026-07-15,2026-07-15,2026-07-18,FABIANE,2026-07-20 00:00:00,2026-07-31 00:00:00,433,NaN
31452,2026,12,VIII,COHAB,COHAB 4,DARCIELE CABRAL/130.177-2,739,COH8114,2026-07-13,2026-07-15,2026-07-15,2026-07-18,FABIANE,2026-07-20 00:00:00,2026-07-31 00:00:00,308,NaN
31453,2026,12,VIII,COHAB,COHAB 4,DARCIELE CABRAL/130.177-2,797,COH8117,2026-07-13,2026-07-15,2026-07-15,2026-07-18,FABIANE,2026-07-20 00:00:00,2026-07-31 00:00:00,95,NaN
31454,2026,12,VIII,COHAB,COHAB 4,DARCIELE CABRAL/130.177-2,825,COH8130,2026-07-13,2026-07-15,2026-07-15,2026-07-18,FABIANE,2026-07-20 00:00:00,2026-07-31 00:00:00,62,NaN


### 1.1 Visão geral dos tipos e valores ausentes

In [4]:
df.info()

print()
print("Valores ausentes por coluna:")
print(df.isna().sum().sort_values(ascending=False))

<class 'pandas.DataFrame'>
RangeIndex: 31455 entries, 0 to 31454
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Ano                 31455 non-null  int64
 1   CICLOS              31455 non-null  int64
 2   DS                  31455 non-null  str  
 3   BAIRRO              31455 non-null  str  
 4   BAIRRO_AREA         31455 non-null  str  
 5   AGENTE / MATRICULA  31455 non-null  str  
 6   QT                  31455 non-null  int64
 7   ID_OVT              31455 non-null  str  
 8   DT_COLETA           30462 non-null  str  
 9   DT_ENTREGA_APOIO    30462 non-null  str  
 10  DT_ENTREGA_LAB      30462 non-null  str  
 11  DT_LEITURA          30462 non-null  str  
 12  NM_TEC_LAB          30462 non-null  str  
 13  DT_DIG              30462 non-null  str  
 14  DT_ENVIO_DS         13691 non-null  str  
 15  N_OVOS              29464 non-null  str  
 16  OBS                 2668 non-null   str  
dtypes: i

## 2. Limpeza dos cabeçalhos

Padronizamos os nomes das colunas para **minúsculas** (snake_case), facilitando o
manuseio no código e no banco.

In [5]:
RENOMEAR = {
    "Ano": "ano", "CICLOS": "ciclo", "DS": "ds", "BAIRRO": "bairro",
    "BAIRRO_AREA": "bairro_area", "AGENTE / MATRICULA": "agente_matricula",
    "QT": "qt", "ID_OVT": "id_ovt", "DT_COLETA": "dt_coleta",
    "DT_ENTREGA_APOIO": "dt_entrega_apoio", "DT_ENTREGA_LAB": "dt_entrega_lab",
    "DT_LEITURA": "dt_leitura", "NM_TEC_LAB": "nm_tecnico_lab",
    "DT_DIG": "dt_digitacao", "DT_ENVIO_DS": "dt_envio_ds",
    "N_OVOS": "n_ovos", "OBS": "obs",
}

df = df.rename(columns=RENOMEAR)
df.columns.tolist()

['ano',
 'ciclo',
 'ds',
 'bairro',
 'bairro_area',
 'agente_matricula',
 'qt',
 'id_ovt',
 'dt_coleta',
 'dt_entrega_apoio',
 'dt_entrega_lab',
 'dt_leitura',
 'nm_tecnico_lab',
 'dt_digitacao',
 'dt_envio_ds',
 'n_ovos',
 'obs']

## 3. Limpeza dos campos de texto

**Problemas encontrados na origem:**

- células compostas apenas por espaços em branco;
- espaços no início/fim de valores (ex.: `nm_tecnico_lab` com `EDILEUZA ` e `EDILEUZA`,
  o que gerava duplicidade de técnicos);
- valores desnecessários como a string `"nan"`.

Aqui removemos os espaços das bordas e convertemos células vazias para ausentes (`NA`).

In [6]:
colunas_texto = [
    "bairro", "bairro_area", "agente_matricula", "id_ovt", "nm_tecnico_lab", "obs",
]

for coluna in colunas_texto:
    df[coluna] = (
        df[coluna]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA})
    )

df["ds"] = df["ds"].astype("string").str.strip()

print("Observação (OBS) após limpeza:")
print(df["obs"].value_counts(dropna=False).to_string())
print()
print("Técnicos de laboratório únicos após limpeza:")
print(df["nm_tecnico_lab"].value_counts(dropna=False).to_string())

Observação (OBS) após limpeza:
obs
<NA>        28787
REC-2026     1579
F             721
E             253
REC-2025      107
R               6
D               2

Técnicos de laboratório únicos após limpeza:
nm_tecnico_lab
ANA CAROLINA    6922
POLLANDRYNNE    5380
WILSON          4419
EDILEUZA        4143
ELAINE          3912
VIRGÍNIA        1915
MARCONI         1501
FABIANE         1257
<NA>             993
PAULO            808
LILIANE          178
EMILLA            27


## 4. Normalização do distrito sanitário (DS)

Na origem o DS aparece em **numeração romana** (`I`, `II`, ... `VIII`). Normalizamos para
valores inteiros `1` a `8`, mais adequados para armazenamento e consulta.

In [7]:
DS_ROMANOS = {
    "I": 1, "II": 2, "III": 3, "IV": 4,
    "V": 5, "VI": 6, "VII": 7, "VIII": 8,
}

df["ds"] = df["ds"].map(DS_ROMANOS)

print("Distritos sanitários encontrados:")
print(df["ds"].value_counts().sort_index().to_string())

Distritos sanitários encontrados:
ds
1    3288
2    5040
3    3852
4    3276
5    3243
6    5100
7    4080
8    3576


## 5. Ajuste dos tipos de dados - datas

As colunas de data chegam em formatos divergentes (só data ou data+horário) e com valores
em branco. Convertemos todas para o tipo `datetime64` com `errors="coerce"`, de forma que
valores inválidos virem ausentes (`NaT`).

In [8]:
colunas_data = [
    "dt_coleta", "dt_entrega_apoio", "dt_entrega_lab", "dt_leitura",
    "dt_digitacao", "dt_envio_ds",
]

# células com espaço em branco viram ausentes antes da conversão
df["dt_envio_ds"] = df["dt_envio_ds"].replace(" ", pd.NA)

for coluna in colunas_data:
    df[coluna] = pd.to_datetime(df[coluna], errors="coerce", format="mixed")

df[colunas_data].dtypes

dt_coleta           datetime64[us]
dt_entrega_apoio    datetime64[us]
dt_entrega_lab      datetime64[us]
dt_leitura          datetime64[us]
dt_digitacao        datetime64[us]
dt_envio_ds         datetime64[us]
dtype: object

## 6. Ajuste dos tipos de dados - números

`n_ovos` (contagem de ovos) e `qt` chegam como texto e precisam ser convertidos para
valores numéricos. A célula `" "` (espaço) existente em `n_ovos` também é tratada como ausente.

In [9]:
df["n_ovos"] = pd.to_numeric(
    df["n_ovos"].astype("string").replace({" ": pd.NA, "": pd.NA}), errors="coerce"
)
df["qt"] = pd.to_numeric(
    df["qt"].astype("string").replace({" ": pd.NA, "": pd.NA}), errors="coerce"
)

print(df[["n_ovos", "qt"]].dtypes.to_string())
print()
print("Estatísticas de n_ovos:")
print(df["n_ovos"].describe().to_string())

n_ovos    Int64
qt        Int64

Estatísticas de n_ovos:
count       29462.0
mean     200.523081
std      238.499652
min             0.0
25%            57.0
50%           128.0
75%           255.0
max          4420.0


## 7. Separação do agente e matrícula

A coluna original `agente_matricula` guarda **nome** e **matrícula** juntos
(ex.: `EMERSON MELO/77.110-0`). Alguns registros têm apenas o nome, e outros usam
`S/M` (= "sem matrícula"). Separamos os campos e normalizamos o `S/M` para ausente.

In [10]:
# Padrão de matrícula "colada" ao nome, sem a barra (ex.: "ÂNGÉLICA BRITO79.718-0")
PADRAO_MATRICULA = re.compile(r"^(.+?)(\d+\.\d+-\d)$")
AGENTES_PLACEHOLDER = {"DESCOBERTA", "ÁREA DESCOBERTA"}


def normalizar_nome(nome):
    """Remove acentos/caixa alta/espaços duplicados (apenas para agrupar)."""
    if not isinstance(nome, str) or not nome:
        return nome
    sem_acento = "".join(
        ch for ch in unicodedata.normalize("NFD", nome)
        if unicodedata.category(ch) != "Mn"
    )
    return " ".join(sem_acento.upper().split())


partes = df["agente_matricula"].str.split("/", n=1, expand=True)
df["agente"] = partes[0].str.strip().str.replace(r"\s+", " ", regex=True)
df["matricula"] = partes[1].astype("string").str.strip().replace("S/M", pd.NA)

# 1) matrícula colada ao nome sem a barra
sem_barra = ~df["agente_matricula"].str.contains("/", na=False)
extraido = df.loc[sem_barra, "agente_matricula"].str.extract(PADRAO_MATRICULA)
df.loc[sem_barra, "agente"] = extraido[0].fillna(df.loc[sem_barra, "agente"])
df.loc[sem_barra, "matricula"] = extraido[1]

# 2) placeholders de "área sem agente responsável" viram ausentes
df.loc[df["agente"].isin(AGENTES_PLACEHOLDER), ["agente", "matricula"]] = pd.NA

# 3) nome normalizado (sem acento) para agrupar grafias do mesmo agente
df["agente_norm"] = df["agente"].map(normalizar_nome)

amostra = df[["agente_matricula", "agente", "matricula", "agente_norm"]].sample(8, random_state=42)
amostra

,agente_matricula,agente,matricula,agente_norm
19973,REGINALDO SILVA/77.460-7,REGINALDO SILVA,77.460-7,REGINALDO SILVA
7102,KLEMERSON/78.998-9,KLEMERSON,78.998-9,KLEMERSON
19911,FRANCISCO ASSIS/77.158-9,FRANCISCO ASSIS,77.158-9,FRANCISCO ASSIS
15777,FABIANA FERNANDA/97.077-5,FABIANA FERNANDA,97.077-5,FABIANA FERNANDA
9805,FÁBIO CLEMENTE/119.225-6,FÁBIO CLEMENTE,119.225-6,FABIO CLEMENTE
8355,MARCELO FRANCISCO/77.353-9,MARCELO FRANCISCO,77.353-9,MARCELO FRANCISCO
20453,MÁRCIA VALÉRIA/77.358-1,MÁRCIA VALÉRIA,77.358-1,MARCIA VALERIA
10824,WANDERLEY SANTANA/77.571-3,WANDERLEY SANTANA,77.571-3,WANDERLEY SANTANA


## 8. Verificação de consistência

Checamos duplicatas totais e a unicidade do par `(id_ovt, ciclo)`, que identifica cada
coleta de uma armadilha.

In [11]:
print("Linhas totalmente duplicadas:", df.duplicated().sum())
print("Par (id_ovt, ciclo) é único?", df.groupby(["id_ovt", "ciclo"]).ngroups == len(df))
print()
print("Valores ausentes após todo o tratamento:")
print(df.isna().sum().sort_values(ascending=False).head(12))

Linhas totalmente duplicadas: 0
Par (id_ovt, ciclo) é único? True

Valores ausentes após todo o tratamento:
obs                 28787
dt_envio_ds         17768
n_ovos               1993
matricula            1289
dt_digitacao          994
nm_tecnico_lab        993
dt_entrega_apoio      993
dt_entrega_lab        993
dt_coleta             993
dt_leitura            993
agente                751
agente_norm           751
dtype: int64


## 9. Organização dos dados em tabelas

Após o tratamento, os dados são organizados em um modelo **dimensional**:

- **dimensões**: `distrito_sanitario`, `bairro`, `agente`, `ovitrampa`,
  `tecnico_laboratorio` e `observacao`;
- **fato**: `monitoramento_ovitrampa` (uma linha por coleta de armadilha).

### 9.1 distrito_sanitario

In [12]:
distrito_sanitario = pd.DataFrame({
    "id_ds": list(range(1, 9)),
    "descricao": [f"Distrito Sanitário {i}" for i in range(1, 9)],
})
distrito_sanitario

,id_ds,descricao
0,1,Distrito Sanitário 1
1,2,Distrito Sanitário 2
2,3,Distrito Sanitário 3
3,4,Distrito Sanitário 4
4,5,Distrito Sanitário 5
5,6,Distrito Sanitário 6
6,7,Distrito Sanitário 7
7,8,Distrito Sanitário 8


### 9.2 bairro

Cada bairro pertence a um único distrito sanitário, então o `DS` vira a chave
estrangeira `id_ds`.

In [13]:
bairro = (
    df[["bairro", "ds"]]
    .dropna(subset=["bairro"])
    .drop_duplicates("bairro")
    .sort_values("bairro")
    .rename(columns={"bairro": "nome", "ds": "id_ds"})
    .reset_index(drop=True)
)
bairro.insert(0, "id_bairro", range(1, len(bairro) + 1))

print("Total de bairros:", len(bairro))
bairro.head()

Total de bairros: 86


,id_bairro,nome,id_ds
0,1,AFLITOS,3
1,2,AFOGADOS,5
2,3,ALTO DO MANDÚ,3
3,4,ALTO JOSÉ BONIFÁCIO,7
4,5,ALTO JOSÉ DO PINHO,7


### 9.3 agente

A matrícula usada é a primeira encontrada para cada nome (agentes aparecem em vários
ciclos, ora com e ora sem matrícula).

In [14]:
base = df[["agente", "agente_norm", "matricula"]].dropna(subset=["agente"]).copy()

# grafia mais frequente dentro de cada nome normalizado = nome canônico
freq = base.groupby("agente_norm")["agente"].agg(lambda s: s.value_counts().index[0])
mapa_norm_canon = freq.to_dict()
mapa_agente_canon = dict(
    zip(
        base[["agente", "agente_norm"]].drop_duplicates()["agente"],
        base[["agente", "agente_norm"]].drop_duplicates()["agente_norm"].map(mapa_norm_canon),
    )
)

agente = (
    base.assign(nome_canonico=base["agente_norm"].map(mapa_norm_canon))
    .groupby("nome_canonico")["matricula"]
    .first()
    .reset_index()
    .sort_values("nome_canonico")
    .rename(columns={"nome_canonico": "nome"})
    .reset_index(drop=True)
)
agente.insert(0, "id_agente", range(1, len(agente) + 1))

print("Total de agentes:", len(agente))
agente.head()

Total de agentes: 678


,id_agente,nome,matricula
0,1,ADENAIDE DÉBORA,76.907-0
1,2,ADILSON FRANCISCO,76.910-4
2,3,ADJANE SILVA,76.911-9
3,4,ADRIANA LOPES,79.621-7
4,5,ADRIANA MARIA,76.914-2


### 9.4 ovitrampa

Cada armadilha (`id_ovt`) pertence a um único bairro; repetida ao longo dos ciclos, ela
entra uma única vez na dimensão.

In [15]:
mapa_bairro = dict(zip(bairro["nome"], bairro["id_bairro"]))

ovitrampa = (
    df[["id_ovt", "bairro"]]
    .dropna(subset=["id_ovt"])
    .drop_duplicates("id_ovt")
    .reset_index(drop=True)
)
ovitrampa["id_bairro"] = ovitrampa["bairro"].map(mapa_bairro)
ovitrampa = ovitrampa[["id_ovt", "id_bairro"]]

print("Total de armadilhas:", len(ovitrampa))
ovitrampa.head()

Total de armadilhas: 2871


,id_ovt,id_bairro
0,REC1001,69
1,REC1002,69
2,REC1003,69
3,REC1004,69
4,REC1005,69


### 9.5 tecnico_laboratorio

In [16]:
tecnico_laboratorio = (
    df[["nm_tecnico_lab"]]
    .dropna()
    .drop_duplicates()
    .sort_values("nm_tecnico_lab")
    .rename(columns={"nm_tecnico_lab": "nome"})
    .reset_index(drop=True)
)
tecnico_laboratorio.insert(0, "id_tecnico", range(1, len(tecnico_laboratorio) + 1))

print("Total de técnicos:", len(tecnico_laboratorio))
tecnico_laboratorio

Total de técnicos: 11


,id_tecnico,nome
0,1,ANA CAROLINA
1,2,EDILEUZA
2,3,ELAINE
3,4,EMILLA
4,5,FABIANE
5,6,LILIANE
6,7,MARCONI
7,8,PAULO
8,9,POLLANDRYNNE
9,10,VIRGÍNIA


### 9.6 observacao

O campo `OBS` guarda códigos (ex.: `F`, `E`, `R`, `D`, `REC-2026`). Criamos uma tabela
de apoio com a **descrição** de cada código (a mesma legenda presente no cabeçalho da origem).

In [17]:
OBSERVACOES = {
    "F": "Fechado",
    "E": "Extraviado",
    "R": "Recusado",
    "D": "Desocupado",
    "REC-2025": "Recuperada - coleção 2025",
    "REC-2026": "Recuperada - coleção 2026",
}

codigos = sorted(df["obs"].dropna().unique())
observacao = pd.DataFrame(
    {"codigo": codigos,
     "descricao": [OBSERVACOES.get(codigo, codigo) for codigo in codigos]}
)
observacao

,codigo,descricao
0,D,Desocupado
1,E,Extraviado
2,F,Fechado
3,R,Recusado
4,REC-2025,Recuperada - coleção 2025
5,REC-2026,Recuperada - coleção 2026


### 9.7 monitoramento_ovitrampa (tabela fato)

A tabela fato registra **cada coleta** (ano, ciclo, armadilha, agente, técnico,
observação, datas e número de ovos), referenciando as dimensões por chaves estrangeiras.

In [18]:
mapa_agente = dict(zip(agente["nome"], agente["id_agente"]))
mapa_tecnico = dict(zip(tecnico_laboratorio["nome"], tecnico_laboratorio["id_tecnico"]))

fato = df.copy()
fato["id_agente"] = fato["agente"].map(mapa_agente_canon).map(mapa_agente)
fato["id_tecnico"] = fato["nm_tecnico_lab"].map(mapa_tecnico)

colunas = [
    "ano", "ciclo", "id_ovt", "id_agente", "id_tecnico", "obs",
    "bairro_area", "qt", "dt_coleta", "dt_entrega_apoio", "dt_entrega_lab",
    "dt_leitura", "dt_digitacao", "dt_envio_ds", "n_ovos",
]
monitoramento = fato[colunas].rename(columns={"obs": "codigo_obs"})
monitoramento.insert(0, "id_registro", range(1, len(monitoramento) + 1))

print("Registros no fato:", len(monitoramento))
monitoramento.head()

Registros no fato: 31455


,id_registro,ano,ciclo,id_ovt,id_agente,id_tecnico,codigo_obs,bairro_area,qt,dt_coleta,dt_entrega_apoio,dt_entrega_lab,dt_leitura,dt_digitacao,dt_envio_ds,n_ovos
0,1,2026,1,REC1001,186.0,1.0,<NA>,RECIFE ANTIGO,1,2026-01-29,2026-01-30,2026-02-05,2026-02-07,2026-03-07,NaT,16
1,2,2026,1,REC1002,186.0,1.0,<NA>,RECIFE ANTIGO,1,2026-01-29,2026-01-30,2026-02-05,2026-02-07,2026-03-07,NaT,14
2,3,2026,1,REC1003,186.0,1.0,<NA>,RECIFE ANTIGO,1,2026-01-29,2026-01-30,2026-02-05,2026-02-07,2026-03-07,NaT,83
3,4,2026,1,REC1004,186.0,1.0,<NA>,RECIFE ANTIGO,1,2026-01-29,2026-01-30,2026-02-05,2026-02-07,2026-03-07,NaT,77
4,5,2026,1,REC1005,186.0,1.0,<NA>,RECIFE ANTIGO,1,2026-01-29,2026-01-30,2026-02-05,2026-02-07,2026-03-07,NaT,88


## 10. Criação das tabelas no SQLite

Definimos o `DDL` (Data Definition Language) com todas as tabelas e suas relações
(primárias e estrangeiras) e criamos o banco `banco/ovitrampas.db`.

In [19]:
DDL = """
DROP TABLE IF EXISTS monitoramento_ovitrampa;
DROP TABLE IF EXISTS observacao;
DROP TABLE IF EXISTS ovitrampa;
DROP TABLE IF EXISTS tecnico_laboratorio;
DROP TABLE IF EXISTS agente;
DROP TABLE IF EXISTS bairro;
DROP TABLE IF EXISTS distrito_sanitario;

CREATE TABLE distrito_sanitario (
    id_ds      INTEGER PRIMARY KEY,
    descricao  TEXT NOT NULL
);

CREATE TABLE bairro (
    id_bairro  INTEGER PRIMARY KEY,
    nome       TEXT NOT NULL UNIQUE,
    id_ds      INTEGER NOT NULL REFERENCES distrito_sanitario (id_ds)
);

CREATE TABLE agente (
    id_agente  INTEGER PRIMARY KEY,
    nome       TEXT NOT NULL UNIQUE,
    matricula  TEXT
);

CREATE TABLE ovitrampa (
    id_ovt     TEXT PRIMARY KEY,
    id_bairro  INTEGER NOT NULL REFERENCES bairro (id_bairro)
);

CREATE TABLE tecnico_laboratorio (
    id_tecnico INTEGER PRIMARY KEY,
    nome       TEXT NOT NULL UNIQUE
);

CREATE TABLE observacao (
    codigo     TEXT PRIMARY KEY,
    descricao  TEXT NOT NULL
);

CREATE TABLE monitoramento_ovitrampa (
    id_registro         INTEGER PRIMARY KEY,
    ano                 INTEGER NOT NULL,
    ciclo               INTEGER NOT NULL,
    id_ovt              TEXT NOT NULL REFERENCES ovitrampa (id_ovt),
    id_agente           INTEGER REFERENCES agente (id_agente),
    id_tecnico          INTEGER REFERENCES tecnico_laboratorio (id_tecnico),
    codigo_obs          TEXT REFERENCES observacao (codigo),
    bairro_area         TEXT,
    qt                  INTEGER,
    dt_coleta           DATE,
    dt_entrega_apoio    DATE,
    dt_entrega_lab      DATE,
    dt_leitura          DATE,
    dt_digitacao        DATE,
    dt_envio_ds         DATE,
    n_ovos              INTEGER,
    UNIQUE (id_ovt, ciclo)
);
"""

conexao = sqlite3.connect(CAMINHO_BANCO)
conexao.execute("PRAGMA foreign_keys = ON")
conexao.executescript(DDL)
conexao.commit()

print("Banco criado:", CAMINHO_BANCO)
print("Tabelas criadas com sucesso.")

Banco criado: C:\Users\pedro\OneDrive\Desktop\Eng_dados_ovitrampas\banco\ovitrampas.db
Tabelas criadas com sucesso.


## 11. Inserção dos dados tratados no banco

As datas são convertidas para o formato ISO (`YYYY-MM-DD`) e nulos tornam-se `NULL`
no banco, garantindo tipos corretos em cada coluna.

In [20]:
def para_insert(tabela: pd.DataFrame) -> list[tuple]:
    """Prepara as linhas do DataFrame para INSERT no SQLite."""
    dados = tabela.astype("object").where(pd.notna(tabela), None)
    registros = []
    for linha in dados.itertuples(index=False, name=None):
        valores = []
        for valor in linha:
            if pd.isna(valor):
                valores.append(None)
            elif hasattr(valor, "date"):
                valores.append(valor.date().isoformat())
            else:
                valores.append(valor)
        registros.append(tuple(valores))
    return registros


tabelas = {
    "distrito_sanitario": distrito_sanitario,
    "bairro": bairro,
    "agente": agente,
    "ovitrampa": ovitrampa,
    "tecnico_laboratorio": tecnico_laboratorio,
    "observacao": observacao,
    "monitoramento_ovitrampa": monitoramento,
}

cursor = conexao.cursor()
for nome, tabela in tabelas.items():
    colunas = ", ".join(tabela.columns)
    marcadores = ", ".join("?" * len(tabela.columns))
    sql = f"INSERT INTO {nome} ({colunas}) VALUES ({marcadores})"
    cursor.executemany(sql, para_insert(tabela))
conexao.commit()
cursor.close()

print("Carga concluída com sucesso!")

Carga concluída com sucesso!


## 12. Validação final

Conferimos a contagem de registros de cada tabela, a integridade referencial e rodamos
uma consulta com junções entre as tabelas.

In [21]:
cursor = conexao.cursor()
print("Registros por tabela:")
for nome in tabelas:
    (quantidade,) = cursor.execute(f"SELECT COUNT(*) FROM {nome}").fetchone()
    print(f"{nome:<26} {quantidade:>6}")

print()
print("Violações de chave estrangeira:", cursor.execute("PRAGMA foreign_key_check").fetchall())
cursor.close()

Registros por tabela:
distrito_sanitario              8
bairro                         86
agente                        678
ovitrampa                    2871
tecnico_laboratorio            11
observacao                      6
monitoramento_ovitrampa     31455

Violações de chave estrangeira: []


In [22]:
consulta = """
SELECT m.ciclo,
       d.id_ds AS ds,
       b.nome  AS bairro,
       o.id_ovt,
       a.nome  AS agente,
       t.nome  AS tecnico,
       m.dt_leitura,
       m.n_ovos,
       ob.descricao AS obs
FROM monitoramento_ovitrampa m
JOIN ovitrampa o              ON o.id_ovt = m.id_ovt
JOIN bairro b                 ON b.id_bairro = o.id_bairro
JOIN distrito_sanitario d     ON d.id_ds = b.id_ds
JOIN agente a                 ON a.id_agente = m.id_agente
LEFT JOIN tecnico_laboratorio t ON t.id_tecnico = m.id_tecnico
LEFT JOIN observacao ob       ON ob.codigo = m.codigo_obs
WHERE m.codigo_obs IS NOT NULL
ORDER BY m.ciclo, m.id_ovt
LIMIT 8
"""

pd.read_sql_query(consulta, conexao)

,ciclo,ds,bairro,id_ovt,agente,tecnico,dt_leitura,n_ovos,obs
0,1,5,AFOGADOS,AFG5004,JONATAN,POLLANDRYNNE,2026-02-24,173.0,Recuperada - coleção 2026
1,1,5,AFOGADOS,AFG5005,SIMONE,POLLANDRYNNE,2026-02-24,NaN,Recuperada - coleção 2025
2,1,5,AFOGADOS,AFG5010,VANDERLUCIA SILVA,WILSON,2026-02-09,NaN,Recuperada - coleção 2025
3,1,5,AFOGADOS,AFG5011,VALNIRA,NaN,NaN,NaN,Fechado
4,1,5,AFOGADOS,AFG5015,JAIRO,POLLANDRYNNE,2026-02-09,NaN,Recuperada - coleção 2025
5,1,5,AFOGADOS,AFG5020,ISAIAS IVO,POLLANDRYNNE,2026-02-24,253.0,Recuperada - coleção 2026
6,1,5,AFOGADOS,AFG5036,VALDEMIR DEMESIO,VIRGÍNIA,2026-03-11,224.0,Recuperada - coleção 2026
7,1,5,AFOGADOS,AFG5037,VALDEMIR DEMESIO,VIRGÍNIA,2026-03-11,282.0,Recuperada - coleção 2026


In [23]:
# ---- Sanity check quantitativo do que foi carregado ----
print("1) Datas fora de ordem (coleta > entrega no apoio):",
      int((monitoramento["dt_coleta"] > monitoramento["dt_entrega_apoio"]).sum()))
print("2) Entrega no lab depois da leitura:",
      int((monitoramento["dt_entrega_lab"] > monitoramento["dt_leitura"]).sum()))
print("3) Leitura depois da digitação:",
      int((monitoramento["dt_leitura"] > monitoramento["dt_digitacao"]).sum()))
print("4) n_ovos > 1000 (provável erro de digitação):",
      int((monitoramento["n_ovos"] > 1000).sum()))
print("5) Leitura registrada sem contagem de ovos:",
      int((monitoramento["dt_leitura"].notna() & monitoramento["n_ovos"].isna()).sum()))
print("6) qt nula ou não positiva:",
      int((monitoramento["qt"] <= 0).sum()), "+", int(monitoramento["qt"].isna().sum()), "nulos")
print("7) Duplicatas (id_ovt, ciclo):",
      int(df.duplicated(["id_ovt", "ciclo"]).sum()))
print("8) Fato sem agente vinculado (fk nula):",
      int(pd.read_sql_query(
          "SELECT COUNT(*) AS n FROM monitoramento_ovitrampa WHERE id_agente IS NULL",
          conexao)["n"].iloc[0]))

1) Datas fora de ordem (coleta > entrega no apoio): 24
2) Entrega no lab depois da leitura: 383
3) Leitura depois da digitação: 17
4) n_ovos > 1000 (provável erro de digitação): 507
5) Leitura registrada sem contagem de ovos: 1002
6) qt nula ou não positiva: 0 + 0 nulos
7) Duplicatas (id_ovt, ciclo): 0
8) Fato sem agente vinculado (fk nula): 751


## Conclusão

Todos os dados do arquivo CSV foram tratados (limpeza de texto, normalização do DS,
tipagem de datas e números, separação agente/matrícula), organizados em tabelas
dimensionais + fato e armazenados com sucesso no banco SQLite
(`banco/ovitrampas.db`), sem perda de registros.

O banco pode agora ser consultado com SQL para responder perguntas como:
- média de ovos por distrito sanitário e ciclo;
- armadilhas com maior infestação (mais ovos) por bairro;
- desempenho dos agentes e técnicos.